# 데이터 전처리


### 라이브러리 설치


In [ ]:
!pip install torch==2.4.0 transformers==4.45.1 datasets==3.0.1 accekerate==0.34.2 trl==0.11.1 peft==0.13.0

In [1]:
from datasets import load_dataset, Dataset
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

c:\workspace\python\rag_master\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


- 데이터셋 로딩 및 변환
  - load_dataset, Dataset: 다양한 데이터셋을 쉽게 로드하고 처리할 수 있습니다. load_dataset은 허깅페이스에 업로드된 데이터셋을 불러오거나 로컬 파일을 로드하는 데 사용하며, Dataset은 개별 데이터셋 객체를 다룰 때 사용합니다.
- 딥러닝 모델과 토크나이저
  - AutoModelForCausalLM: 허깅페이스 Transformers 라이브러리에서 제공하는 모델 다운로드를 위한 도구입니다. 언어 모델을 로드하는데 사용합니다.
  - AutoTokenizer: 특정 모델에 맞는 토크나이저를 자동으로 불러오는 도구입니다. 토크나이저는 텍스트를 언어 모델이 처리할 수 있는 정수 시퀀스로 변환하거나 정수 시퀀스를 다시 텍스트 문자열로 복원하는 역할을 합니다.
- 파인튜닝 및 효율적인 학습 구성
  - LoraConfig: 이번 실습에서 사용할 학습 방법인 LoRA(Low-Rank Adaptation) 학습 방식을 사용할 때 필요한 각종 설정값을 정의합니다. LoRA는 학습할 때 대규모 언어 모델 전체를 업데이트하는 것이 아니라, 대규모 언어 모델의 특정 부분만 업데이트하여 보다 효율적으로 학습하는 방식입니다.
- SFT(지도 학습 방식의 파인튜닝) 설정 및 학습도구
  - SFTConfig: 모델을 학습할 때 필요한 다양한 설정값을 정의하는 도구입니다. 학습과정에서 모델을 어떻게 업데이트할지 조정하며 여기에서는 학습률과 배치 크기, 옵티마이저등의 설정이 포함됩니다. 모델 전체를 학습할 때도 쓰일 수 있지만, 이번 실습에 사용하는 LoRA 학습에서처럼 모델의 특정 부분만 학습하는 방식에서도 사용합니다. SFTConfig에서 설정하는 모델의 학습 성능과 안정성에 큰 영향을 줍니다.
  - 예를들어 학습률이 너무 크면 학습이 불안정하고, 너무 작으면 속도가 느려집니다. 배치크기는 한번에 처리하는 데이터 개수를 결정하며 크기에 따라 학습 안정성과 메모리 사용량이 달라집니다. 옵티마이저는 모델을 업데이트하는 방식과 관련된 것으로, Adam SGD등 여러 종류가 있으며 학습 성능을 좌우합니다.
  - SFTConfig에는 이 외에도 가중치 감쇠, 학습 스케쥴링, 혼합 정밀도 학습(fp16) 등 학습에 영향을 주는 다양한 설정을 포함할 수 있습니다. LoRA를 적용하면 모델 전체가 아닌 일부 가중치만 학습하므로 LoRA와 연관된 설정은 LoraConfig에서 관리하지만, LoRA 학습과는 별개인 학습 과정 전반의 설정은 여전히 SFTConfig에서 관리합니다. 따라서 SFTConfig는 LoRAConfig와 함께 사용하며 학습을 조정하는 역할을 합니다.
- SFTTrainer: 실제 학습을 수행하는 클래스입니다. 주어진 데이터셋을 이용해 파일튜닝 과정을 자동으로 수행하며, 특정 부분만 업데이트하는 LoRA와 같은 학습 기봅도 적용할 수 있습니다. 모델, 데이터셋, 학습 설정을 한 번에 입력하여 효율적인 학습을 진행할 수 있도록 돕습니다.


In [3]:
# 허깅페이스 허브에서 데이터셋 로드
dataset = load_dataset("iamjoon/klue-mrc-ko-rag-dataset", split="train")

# system_message 정의
system_message = """당신은 검색 결과를 바탕으로 질문에 답변해야 합니다.

다음의 지시사항을 따르십시오.
1. 질문과 검색 결과를 바탕으로 답변하십시오.
2. 검색 결과에 없는 내용을 답변하려고 하지 마십시오.
3. 질문에 대한 답이 검색 결과에 없다면 검색 결과에는 "해당 질문~에 대한 내용이 없습니다."라고 답변하신시오.
4. 답변할 때 특정 문서를 참고하여 문장 또는 문단을 작성했다면 뒤에 출처는 이중 리스트로 해당 문서 번호를 남기십시오.
  예를 들어 특정 문장이나 문단을 1번 문서에서 인용했다면 뒤에 [[ref1]]이라고 기재하십시오.
5. 예를 들어 특정 문장이나 문단을 1번 문서와 5번 문서에서 동시에 인용했다면 뒤에 [[ref1]], [[ref5]]라고 기재하십시오.
6. 최대한 다수의 문서를 인용하여 답변하십시오.

검색 결과:
-----
{search_result}"""

# 원본 데이터의 type별 분포 출력
print("원본 데이터의 type 분포:")
for type_name in set(dataset["type"]):
    print(f"{type_name}: {dataset['type'].count(type_name)}개")

# train/test 분할 비율 설정(0.5면 5:5로 분할)
test_ratio = 0.8

train_data = []
test_data = []

# type별로 순회하면서 train/test 데이터 분할
for type_name in set(dataset["type"]):
    # 현재 type에 해당하는 데이터 인덱스만 추출
    curr_type_data = [i for i in range(len(dataset)) if dataset[i]["type"] == type_name]

    # test_ratio에 따라 test 데이터 개수 계산
    test_size = int(len(curr_type_data) * test_ratio)

    # 현재 type의 데이터를 test_ratio 비율로 분할하여 추가
    test_data.extend(curr_type_data[:test_size])
    train_data.extend(curr_type_data[test_size:])


# OpenAI format으로 데이터를 변환하기 위한 함수
def format_data(sample):
    # 검색 결과를 문서1, 문서2... 형태로 포매팅
    search_result = "\n-----\n".join(
        [
            f"문서{idx + 1}: {result}"
            for idx, result in enumerate(sample["search_result"])
        ]
    )

    # OpenAI forma로 변환
    return {
        "messages": [
            {
                "role": "system",
                "content": system_message.format(search_result=search_result),
            },
            {"role": "user", "content": sample["question"]},
            {"role": "assistant", "content": sample["answer"]},
        ]
    }


# 분할된 데이터를 OpenAI format으로 변환
train_dataset = [format_data(dataset[i]) for i in train_data]
test_dataset = [format_data(dataset[i]) for i in test_data]

# 최종 데이터셋 크기 출력
print(
    f"\n전체 데이터 분할 결과: Train: {len(train_dataset)}개, Test: {len(test_dataset)}개"
)

# 분할된 데이터의 type별 분포 출력
print("\n학습 데이터의 type 분포:")
for type_name in set(dataset["type"]):
    count = sum(1 for i in train_data if dataset[i]["type"] == type_name)
    print(f"{type_name}: {count}개")

print("\n테스트 데이터의 type 분포:")
for type_name in set(dataset["type"]):
    count = sum(1 for i in test_data if dataset[i]["type"] == type_name)
    print(f"{type_name}: {count}개")


원본 데이터의 type 분포:
mrc_question_with_1_to_4_negative: 296개
paraphrased_question: 196개
synthetic_question: 497개
mrc_question: 491개
no_answer: 404개

전체 데이터 분할 결과: Train: 380개, Test: 1504개

학습 데이터의 type 분포:
mrc_question_with_1_to_4_negative: 60개
paraphrased_question: 40개
synthetic_question: 100개
mrc_question: 99개
no_answer: 81개

테스트 데이터의 type 분포:
mrc_question_with_1_to_4_negative: 236개
paraphrased_question: 156개
synthetic_question: 397개
mrc_question: 392개
no_answer: 323개


1. 데이터셋 로드

- 허깅페이스 허브에서 데이터 셋을 불러옵니다. load_dataset() 함수를 사용하여 데이터셋을 불러옵니다.


2. 시스템 프롬프트 정의

- 학습에서 사용할 시스템 프롬프트를 정의합니다. 이 프롬프트에는 RAG 성능을 높이기 위한 여러가지 중요한 지침이 담겨있습니다.
- 첫째. 검색결과를 바탕으로 답변을 생성하도록 합니다.
- 둘째. 검색결과에 없는 내용으로 답변하지 말라는 제약을 둡니다.
- 셋째. 특정 질문에 대한 내용이 검색결과에 없을 경우 그 사실을 명시적으로 알리도록 합니다.
- 넷째. 답변시 참고한 문서는 반드시 [[ref1]]과 같은 형식으로 표시하도록 요구합니다. 여러 문서를 참고한 경우 [[ref1]], [[ref5]]와 같이 모든 참고 문서를 표시하도록 합니다.
- 마지막: 가능한 한 많은 문서를 참고하여 답변하도록 지시합니다.
- 프롬프트 끝의 {search_result}는 추후 실제 검색 결과로 대체됩니다. 학습할 때도 우리가 원하는 방향으로 대규모 언어 모델이 답변하도록 상세한 시스템프롬프트를 작성해야 합니다.


3. 원본 데이터 타입 분포 확인

- 원본 데이터의 type별 분포를 확인합니다.
- set() 함수로 중복없는 type 목록을 만들고, count() 메서드로 각 type이 몇 번 등장하는지 계산해 출력합니다.
- 데이터셋에는 앞에서 설명했듯이 synthetic_question, mrc_question, mrc_question_with_1_to_4_negative, paraphrased_question, no_answer라는 5가지 타입이 있습니다.


4. 학습용/테스트용 데이터 분할 비율 설정

- train/test 데이터 분할 비율을 설정합니다.
- test_ratio 변수에 0.8을 할당하여 전체 데이터의 80%를 테스트 데이터로, 나머지 20%를 학습데이터로 사용하도록 지정합니다.
- 보통은 학습 데이터의 양이 더 많고, 성능을 평가하기 위한 테스트 데이터의 양이 더 적습니다.
- 유료 클라우드를 사용하므로, 과도한 학습 비용을 방지하기 위해서 학습데이터를 적게 설정했습니다.
- 분할된 데이터의 인덱스를 저장할 train_data와 test_data라는 빈 리스트를 생성합니다.


5. 타입별 데이터 분할

- 데이터셋의 균형을 유지하면서 학습용과 테스트용 데이터를 분리하는 부분입니다.
- 각 타입별로 동일한 비율로 분할하는 이유는, 무작위로 전체 데이터를 나누면 특정 타입의 데이터가 한쪽으로 쏠릴 수 있기 때문입니다.


6. OpenAI 형식으로 데이터 변환 함수 정의

- OpenAI 형식으로 데이터를 변환하는 format_data() 함수를 정의합니다.
- message라는 리스트 안에 각각의 대화를 역할과 내용으로 구분하여 담는 구조입니다.


7. 분할된 데이터를 OpenAI 형식으로 변환

- 앞서 분할한 train_data와 test_data의 각 샘플에 format_data() 함수를 적용하여 최종 데이터셋을 생성합니다.
- 각 인덱스에 해당하는 데이터를 format_data() 함수를 이용해 OpenAI 형식으로 변환하여 train_dataset과 test_dataset에 저장합니다.


8. 최종 데이터셋 크기 출력

- 최종적으로 만들어진 train_dataset과 test_dataset의 크기를 출력합니다. 이를 통해 데이터 분할이 의도한 대로 이루어졌는지 확인할 수 있습니다.


9. 분할된 데이터 타입별 분포 출력

- 분할된 데이터의 type별 분포를 출력합니다. 학습 데이터와 테스트 데이터 각각에 대해 type별 개수를 계산하고 출력합니다.
- 이를 통해 데이터 분할이 각 type에 대해 균형 있게 이루어졌는지 검증할 수 있습니다.


### OpenAI 형식 확인하기

- 임의로 345번 샘플을 출력해봅니다.


In [4]:
train_dataset[345]["messages"]

[{'role': 'system',
  'content': '당신은 검색 결과를 바탕으로 질문에 답변해야 합니다.\n\n다음의 지시사항을 따르십시오.\n1. 질문과 검색 결과를 바탕으로 답변하십시오.\n2. 검색 결과에 없는 내용을 답변하려고 하지 마십시오.\n3. 질문에 대한 답이 검색 결과에 없다면 검색 결과에는 "해당 질문~에 대한 내용이 없습니다."라고 답변하신시오.\n4. 답변할 때 특정 문서를 참고하여 문장 또는 문단을 작성했다면 뒤에 출처는 이중 리스트로 해당 문서 번호를 남기십시오.\n  예를 들어 특정 문장이나 문단을 1번 문서에서 인용했다면 뒤에 [[ref1]]이라고 기재하십시오.\n5. 예를 들어 특정 문장이나 문단을 1번 문서와 5번 문서에서 동시에 인용했다면 뒤에 [[ref1]], [[ref5]]라고 기재하십시오.\n6. 최대한 다수의 문서를 인용하여 답변하십시오.\n\n검색 결과:\n-----\n문서1: “우버(UBER)는 사람들이 이동하는 수단을 발전시켰습니다. 승객과 기사를 실시간으로 연결해 승객에게는 편리함을, 기사에게는 더 많은 효율성과 수익을 가져다 주었습니다. 2010년 6월 미국 샌프란시스코에서 처음 서비스를 시작한 뒤 현재 세계 140여개 도시에 진출했습니다. 우버는 앞으로도 계속 사람과 도시를 가깝게 이어줄 것입니다.”샌프란시스코 우버 본사에서 최근 만난 나이리 후다지안 우버 글로벌커뮤니케이션 부문장은 이렇게 강조했다. 우버는 일종의 ‘차량 예약 서비스’를 제공하는 글로벌 정보기술(IT) 기업이다. 스마트폰 애플리케이션(앱·응용프로그램·사진)을 통해 승객과 차량을 연결해 주는 사업을 하고 있다.승객을 일반택시와 연결해 주는 ‘우버택시’, 일반인이 자신의 차량으로 운송 서비스를 할 수 있도록 도와주는 ‘우버엑스’, 일종의 고급 콜택시인 ‘우버블랙’ 등의 서비스를 갖췄다. 한국에서는 지난해 8월 우버코리아가 설립돼 우버블랙 사업을 펼치고 있다.후다지안 부문장은 “우리는 더 나은 교통 서비스 위해 플랫폼을 제공하는 회사”라고

- role이 system인 경우, content에는 앞에서 작성한 시스템 프롬프트와 현재 샘플의 검색 결과가 저장되어 있습니다.
- role이 assistant인 경우, content에는 검색 결과와 사용자 질문을 바탕으로 대규모 언어 모델이 답변해야 할 내용이 작성되어 있습니다.
- OpenAI 형식은 학습하기 위한 최종 형식은 아니며, 전처리를 위한 중간 단계 형태 입니다.
- 학습에 사용하는 최종 형식은 뒤에서 다를 대규모 언어 모델의 토크나이저를 통해 한 번 더 전처리를 진행하고 나서 결정됩니다.


### 데이터 타입 변경

- 현재 train_dataset과 test_dataset은 데이터 타입이 리스트입니다.
- 원할하게 학습을 진행하려면 데이터 타입을 Dataset으로 변경해야 합니다.


In [ ]:
# 리스트 형태에서 다시 Dataset 형태로 변환
print(type(train_dataset))
print(type(test_dataset))

train_dataset = Dataset.from_list(train_dataset)
test_dataset = Dataset.from_list(test_dataset)

print(type(train_dataset))
print(type(test_dataset))